Step 1: Split the received signal as ref signal \& loc signal. Get the basic information including sample rate, starting time, etc, and save it to folder with name $date/

In [ ]:
from utils.calibration import CalibrationSignal
import os
from matplotlib import pyplot as plt
import scipy.io as sio
import numpy as np
iq0_path ='/home/ishani/LeoPNT/USRP_data/2_5/utctime_5Feb2024_23:41:22_freq_11_325G_bw_2.0M_ts_1707176482.0819871_inrx0.dat'
iq1_path = iq0_path.replace('inrx0', 'inrx1')
iq2_path = iq0_path.replace('inrx0', 'inrx2')
date = "2_5"
start_time = float(iq0_path.split('_')[-2])
fc = 11.325e9
tx_freq = 11.3251e9
sample_rate = 2e6
cal0 = CalibrationSignal(iq0_path, fc, tx_freq, sample_rate) # creates an instance of CalibrationSignal. More details in utils/calibration.py 
cal1 = CalibrationSignal(iq1_path, fc, tx_freq, sample_rate)
cal2 = CalibrationSignal(iq2_path, fc, tx_freq, sample_rate)
ref_sig_idx = cal0.get_all_refsigidx() # use the get_all_refsigidx() method to get the start and stop index of the reference signal
phase_0 = cal0.get_refsig_phase(ref_sig_idx)
phase_1 = cal1.get_refsig_phase(ref_sig_idx)
phase_2 = cal2.get_refsig_phase(ref_sig_idx)
phase_offset_01 = np.unwrap(phase_0-phase_1)
phase_offset_21 = np.unwrap(phase_2-phase_1)
t_end = ref_sig_idx[-1][1]/sample_rate  # use the stop time of the last ref sig as the duration of the file

if not os.path.exists(date):
    os.makedirs(date)
# save sample_rate, date, start_time, loc_sig_idx to .mat file
sio.savemat(date + '/info.mat', {'IQ_Sample_file_0': iq0_path,'sample_rate': sample_rate, 'date': date, 'start_time': start_time, 'ref_sig_idx': ref_sig_idx, 'phase_offset_01': phase_offset_01, 'phase_offset_21': phase_offset_21, 't_end': t_end})
%matplotlib widget 
plt.close('all')
plt.plot(np.unwrap(phase_0-phase_1), label='rx0    -    rx1')
plt.plot(np.unwrap(phase_2-phase_1), label='rx2    -    rx1')
plt.xlabel('ref sig #')
plt.legend(loc='upper right')
plt.show()


Step 2: do the central tone tracking using matlab

In [ ]:
# need to change date first in matlab file
# matlab
# cd Starloc/
# phase_from_bin_track
# results saved to $date/res.mat

Step 3: have a look at the results

In [ ]:
from matplotlib import pyplot as plt
from scipy import io as sio
date = "2_5"
res = sio.loadmat(date + '/res.mat')['res']
plt.close('all')
%matplotlib widget
plt.plot(res[:, 0], res[:, 1], '.')
plt.show()

Step 4: simulate doppler, first need to move the tle file to the date folder with name `Starlink.tle`

In [ ]:
from sat_id import connect_doppler, doppler_sim
date = "2_5"
sim_start_time = start_time
sim_end_time = sim_start_time+t_end
sat_dopp_dict, t_list = doppler_sim(date, date+'/Starlink.tle', fc, sim_start_time, sim_end_time, 1)
output_doppler_file = date+"/output_doppler.mat"
output = {}

sio.savemat(output_doppler_file, {'sat_dopp_dict': sat_dopp_dict, 't_list': t_list})

Step 5: Run `phase_from_bin.m` to get phase difference.

Step 6: Do localization based on phase diff

In [ ]:
from AoA_formula import calc_AoA_2
import numpy as np
d = 13.25e-2
lam = 3e8/fc
from matplotlib import pyplot as plt
data = sio.loadmat(date+'/phi_test.mat')
del_phi_01_gt = data['del_phi_01_gt'][0]
del_phi_12_gt = data['del_phi_12_gt'][0]
del_phi_01 = data['del_phi_01_LS'][0]
del_phi_12 = data['del_phi_12_LS'][0]
az_gt = data['az_gt'][0]
el_gt = data['el_gt'][0]
t_list = data['t_sim'][0]
plt.plot(del_phi_01_gt)
plt.plot(del_phi_12_gt)
plt.show()
az_gt_amb, el_gt_amb, num_amb_gt = calc_AoA_2(del_phi_01_gt, del_phi_12_gt, d, d, lam, 0, 0)
az_amb, el_amb, num_amb = calc_AoA_2(-del_phi_01, -del_phi_12, d, d, lam, 0, 0)
for i in range(num_amb_gt):
    plt.plot(np.rad2deg(el_gt_amb[i,:]), '--')
plt.plot(el_gt)
plt.show()
for i in range(num_amb_gt):
    plt.plot(np.rad2deg(az_gt_amb[i,:]), '--')
plt.plot(az_gt)
plt.show()

In [ ]:
def loc_aoa(date, az, el, t_list, t_start, lat_gt, lon_gt, sat_name):
    from skyfield.api import load, wgs84
    from scipy.optimize import minimize
    from tqdm import tqdm
    from datetime import datetime, timedelta
    import pytz
    # Function to calculate the difference between observed and calculated azimuth and altitude
    def difference(params, satellite, gt_azimuth, gt_elevation, t):
        observer = wgs84.latlon(params[0], params[1])
        dist = (satellite-observer).at(t)
        alt, az, _ = dist.altaz()
        return np.linalg.norm([alt.degrees-gt_elevation, az.degrees-gt_azimuth])

    # Initial guess for observer's latitude and longitude
    satellites = load.tle_file(date+'/Starlink.tle')
    for satellite in satellites:
        if satellite.name == sat_name:
            sat = satellite
            break
    
    lat_est, lon_est = np.empty((num_amb_gt, len(t_list))), np.empty((num_amb_gt, len(t_list)))
    t_start += 6*3600
    el_new_sim = np.empty(len(t_list))
    for i in range(len(az)):
        initial_guess = [lat_gt, lon_gt]
        bounds = [(-89.9, 89.9), (-179.9, 179.9)]
        ts = load.timescale()
        for j in tqdm(range(len(t_list))):
            d_1 = datetime.fromtimestamp(t_start+t_list[j])
            t_dt = datetime(d_1.year, d_1.month, d_1.day, d_1.hour, d_1.minute, d_1.second, tzinfo=pytz.UTC)
            t = ts.utc(t_dt)

            observer = wgs84.latlon(lat_gt, lon_gt)
            dist = (sat-observer).at(t)
            alt_, az_, _ = dist.altaz()
            el_new_sim[j] = np.deg2rad(alt_.degrees)
                
            result = minimize(difference, initial_guess, args=(sat, np.rad2deg(az[i, j]), np.rad2deg(el[i, j]), t), method='L-BFGS-B', bounds=bounds)
            lat_est[i, j], lon_est[i, j] = result.x
            initial_guess = result.x
    return lat_est, lon_est, el_new_sim

lat_est, lon_est, el_new = loc_aoa(date, az_gt_amb, el_gt_amb, t_list, start_time, +40.072969193609126, -88.20958051030674, 'STARLINK-6079')

In [ ]:
for i in range(num_amb_gt):
    plt.plot(lat_est[i,:], lon_est[i,:], '.')

plt.show()